# BUMP OF CHICKENは今までに何曲作ったの？

**氏名：［神郡正稀］**

## 調べたいこと

私の好きなバンド「BUMP OF CHICKEN」の楽曲について<br>
BUMP OF CHICKENの曲がそれぞれいくつデジタルリリースされているかが分からないので調べてみたい。<br>
出典：[iTunes Search API](https://developer.apple.com/library/archive/documentation/AudioVideo/Conceptual/iTuneSearchAPI/)／取得日：2026年10月21日／時刻：日本時間

## 1.BUMPの曲を検索

課題の例の方法だと上位200件までの曲しか検索できないことが分かったので、検索方法を工夫しました。<br>
リリースされたアルバムごとに曲を検索する方法でトライ。

In [1]:
import json
import time
from urllib.request import urlopen

url = "https://itunes.apple.com/search?term=BUMP+OF+CHICKEN&entity=song&country=JP&limit=50"

with urlopen(url, timeout=20) as response:
    data = json.load(response)
artist_id = data["results"][0]["artistId"]

#アーティストのアルバム一覧を取る
url = f"https://itunes.apple.com/lookup?id={artist_id}&entity=album&country=JP&limit=200"

with urlopen(url, timeout=20) as response:
    albums = [a for a in json.load(response)["results"] if a["wrapperType"] == "collection"]
print("アルバム数:", len(albums))

#アルバムごとに曲を取る
all_songs = {}
for album in albums:
    url = f"https://itunes.apple.com/lookup?id={album['collectionId']}&entity=song&country=JP&limit=200"
    with urlopen(url, timeout=20) as response:
        for s in json.load(response)["results"]:
            if s["wrapperType"] == "track":
                all_songs[s["trackId"]] = s
    time.sleep(4)   # 1分に約20回までの目安を守る

songs = list(all_songs.values())
print("曲数:", len(songs)) 

アルバム数: 71
曲数: 377


5分ほど時間が掛かってしまいましたが、検索することに成功しました。<br>

## 2.曲の分別

さっきの377曲の内、アルバムとシングルに収録されている曲もアルバム版とシングル版とで登録されています。<br>
このダブりを消すためにシングル盤の曲の中で、同じ曲名がアルバム側にもあるものを除くようにしました。

In [2]:
def is_single(s):
    return "- Single" in s.get("collectionName", "")

def is_live(s):
    name = s.get("collectionName", "").upper()
    return "LIVE" in name or "TOUR" in name

# シングルでもライブでもない盤(アルバムなど)に入っている曲名
album_names = {s["trackName"] for s in songs if not is_single(s) and not is_live(s)}

#シングル盤の曲のうち、同じ曲名がアルバム側にもあるものを除く
cleaned = [s for s in songs if not (is_single(s) and s["trackName"] in album_names)]

print("除く前:", len(songs), "→ 除いた後:", len(cleaned))

除く前: 377 → 除いた後: 309


無事に数曲が除かれました。<br>
念のため、曲名のダブりがないか検索します。

In [3]:
live = [s for s in cleaned if is_live(s)]
normal = [s for s in cleaned if not is_live(s)]

print("通常曲 収録数:", len(normal), "/ 曲名の種類:", len({s["trackName"] for s in normal}))
print("LIVE版 収録数:", len(live), "/ 曲名の種類:", len({s["trackName"] for s in live}))

通常曲 収録数: 176 / 曲名の種類: 150
LIVE版 収録数: 133 / 曲名の種類: 133


ベストアルバムを除外し忘れていました。<br>
しかし、これでBUMPが今までに出した曲数（曲名の種類）が分かりました。また、デジタル化されているLIVE曲の曲数も分かりました。

## 3.最後に　自己満足コード

最後にせっかくなので、ランダムに曲を並べて見れるようにしました。<br>
下のコードの`N = 数字`の数字部分を変えることで表示される曲数が変わります。

In [4]:
import random

N = 10
for s in random.sample(normal, N):
    date = s["releaseDate"][:10].replace("-", "/")
    print(s["trackName"], "/", s["collectionName"], "/", date)

くだらない唄 / BUMP OF CHICKEN I [1999-2004] / 1999/03/18
サザンクロス / RAY / 2014/03/12
Smile / RAY / 2011/05/11
ジャングルジム / aurora arc / 2019/07/10
ray (feat. 初音ミク) / ray (feat. 初音ミク) - Single / 2014/03/12
strawberry / Iris / 2024/09/04
Title of mine / jupiter / 2002/02/20
ゼロ / RAY / 2011/10/19
Ending / THE LIVING DEAD / 2000/03/25
ガラスのブルース / FLAME VEIN +1 / 1999/03/18


## 分かったこと

BUMP OF CHICKENのリリースされた通常曲は150種類、LIVE版は133種類ありました。

**確認：** 出典が開けることを確認しました。また、曲名、アルバム名に誤字、リリース日に誤りがないことを確認しました。

**限界：** BUMPのアルバムには「隠しトラック」があるのですが、それらはデジタルリリースされていないため曲数に含まれていません。どう頑張ってもデジタル（iTunes Store）配信されていない曲の情報は得られないです。

ps.AIに自分のやりたいことが技術的に可能かを相談しました。また、コードの一つ一つを自分でも理解できるように説明してもらいました。

## 通信できないときだけ

保存データ`bump-songs.json`を同じフォルダに置き、最初のCode Cellを次のコードに置き換えます。2026年10月21日の保存例なので、最新のデータではありません。

```python
import json

with open("bump-songs.json", encoding="utf-8") as file:
    songs = json.load(file)
```